# 0. feladat

In [2]:
# Imports:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import torch.optim as optim
import torch.nn as nn
import pandas as pd
import torch
import json

from IDS import IntrusionDetector


In [3]:
# Loading from csv
with open('nslkdd_features.json', 'r') as f:
    features = json.load(f)
col_names = [feature['name'] for feature in features]
df_train_from_csv = pd.read_csv('NSL-KDD/KDDTrain+.txt', names=col_names)
df_test_from_csv = pd.read_csv('NSL-KDD/KDDTest+.txt', names=col_names)

# normal: 0, attack: 1
train_labels = (df_train_from_csv['label'] != 'normal').astype(int)
test_labels = (df_test_from_csv['label'] != 'normal').astype(int)


X_train_raw = df_train_from_csv.drop(['label', 'difficulty'], axis=1)
X_test_raw = df_test_from_csv.drop(['label', 'difficulty'], axis=1)

X_train_raw['is_train'] = 1
X_test_raw['is_train'] = 0

# Combine into one dataframe to get the 122 features, without this, 
# we would get different one-hot encoded features for train and test sets
combined_df = pd.concat([X_train_raw, X_test_raw], ignore_index=True)


# One-hot encoding
categorical_cols = ['protocol_type', 'service', 'flag']
combined_df = pd.get_dummies(combined_df, columns=categorical_cols)

# Split the data back into train and test sets
X_train = combined_df[combined_df['is_train'] == 1]
X_test= combined_df[combined_df['is_train'] == 0]
X_train.drop('is_train', axis=1, inplace=True)
X_test.drop('is_train', axis=1, inplace=True)


# StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print(X_train.shape)
print(X_test.shape)



X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(train_labels, dtype=torch.float32).unsqueeze(1) # 1D -> 2D torch need this
y_test_tensor = torch.tensor(test_labels, dtype=torch.float32).unsqueeze(1)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

C:\Users\Szautner_Bela\AppData\Local\Temp\ipykernel_123488\2348304070.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop('is_train', axis=1, inplace=True)
C:\Users\Szautner_Bela\AppData\Local\Temp\ipykernel_123488\2348304070.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_test.drop('is_train', axis=1, inplace=True)


(125973, 122)
(22544, 122)


# 1. feladat


In [4]:
# Trainning the model
model = IntrusionDetector(input_dim = X_train.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
batch_size=128
epochs=10
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

for epoch in range(epochs):
    for features, label in train_loader:
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()

torch.save(model.state_dict(), "trained_model.pth")

In [5]:
# Evaluation 
model.eval()

all_preds = []
all_targets = []

wrongly_classified_test_data = []

# Zero-day attack evaluation
with torch.no_grad():
    for i, (features, label) in enumerate(test_dataset):
        X_batch = features.to('cpu')
        y_batch = label.to('cpu')

        y_pred_probs = model(X_batch)
        
        y_pred_binary = (y_pred_probs >= 0.5).float()
        if y_pred_binary.item() != y_batch.item():
            wrongly_classified_test_data.append(i)

        all_preds.extend(y_pred_binary.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())
        
# 4. Calculate metrics over the ENTIRE dataset
accuracy = accuracy_score(all_targets, all_preds)
precision = precision_score(all_targets, all_preds)
recall = recall_score(all_targets, all_preds)
f1 = f1_score(all_targets, all_preds)

# 5. Print out the results
print("Zero-day attack evaluation:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")



all_preds_on_train = []
all_targets_on_train = []

# Non zero-day attack evaluation
with torch.no_grad():
    for features, label in train_dataset:
        X_batch = features.to('cpu')
        y_batch = label.to('cpu')

        y_pred_probs = model(X_batch)
        
        y_pred_binary = (y_pred_probs >= 0.5).float()

        all_preds_on_train.extend(y_pred_binary.cpu().numpy())
        all_targets_on_train.extend(y_batch.cpu().numpy())
        
# 4. Calculate metrics over the ENTIRE dataset
accuracy = accuracy_score(all_targets_on_train, all_preds_on_train)
precision = precision_score(all_targets_on_train, all_preds_on_train)
recall = recall_score(all_targets_on_train, all_preds_on_train)
f1 = f1_score(all_targets_on_train, all_preds_on_train)

# 5. Print out the results
print("\nNon zero-day attack evaluation:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Zero-day attack evaluation:
Accuracy:  0.7900
Precision: 0.9290
Recall:    0.6833
F1 Score:  0.7874

Non zero-day attack evaluation:
Accuracy:  0.9951
Precision: 0.9969
Recall:    0.9925
F1 Score:  0.9947


# 2. feladat

In [52]:
# ============================================================
# TASK 2 - PGD ATTACK
# ============================================================

import sys
sys.path.append('.')
from simple_rules import AttackPlausibilityChecker

# 2.1 - Feature names after one-hot encoding
X_train_raw_temp = df_train_from_csv.drop(['label', 'difficulty'], axis=1)
X_test_raw_temp = df_test_from_csv.drop(['label', 'difficulty'], axis=1)
X_train_raw_temp['is_train'] = 1
X_test_raw_temp['is_train'] = 0
combined_temp = pd.concat([X_train_raw_temp, X_test_raw_temp], ignore_index=True)
combined_temp = pd.get_dummies(combined_temp, columns=['protocol_type', 'service', 'flag'])
feature_names_ohe = [col for col in combined_temp.columns if col != 'is_train']
print(f"Number of features: {len(feature_names_ohe)}")

# 2.2 - Extended modifiable feature indices (highly + partially)
partially_modifiable_simple = [
    'dst_bytes', 'hot', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate'
]
modifiable_indices_extended = []
modifiable_names_extended = []
for fname in ['duration', 'src_bytes', 'wrong_fragment', 'urgent'] + partially_modifiable_simple:
    if fname in feature_names_ohe:
        idx = feature_names_ohe.index(fname)
        modifiable_indices_extended.append(idx)
        modifiable_names_extended.append(fname)
print(f"Modifiable features: {len(modifiable_indices_extended)}")

# 2.3 - Feature min/max in standardized space (computed from training data)
feature_min = torch.tensor(X_train.min(axis=0), dtype=torch.float32)
feature_max = torch.tensor(X_train.max(axis=0), dtype=torch.float32)

# 2.4 - Identify correctly detected attack samples
model.eval()
correctly_detected_attack_indices = []
with torch.no_grad():
    for i, (features, label) in enumerate(test_dataset):
        if label.item() == 1:
            pred = model(features).item()
            if pred >= 0.5:
                correctly_detected_attack_indices.append(i)
print(f"Correctly detected attacks: {len(correctly_detected_attack_indices)}")

# 2.5 - Attack type mapping
dos_attacks = ['back', 'land', 'neptune', 'pod', 'smurf', 'teardrop',
               'apache2', 'udpstorm', 'processtable', 'worm', 'mailbomb']
probe_attacks = ['satan', 'ipsweep', 'nmap', 'portsweep', 'mscan', 'saint']
r2l_attacks = ['ftp_write', 'guess_passwd', 'imap', 'multihop', 'phf', 'spy',
               'warezclient', 'warezmaster', 'sendmail', 'named', 'snmpgetattack',
               'snmpguess', 'xlock', 'xsnoop', 'httptunnel']
u2r_attacks = ['buffer_overflow', 'loadmodule', 'perl', 'rootkit', 'sqlattack',
               'xterm', 'ps']

def get_attack_type(label_str):
    if label_str in dos_attacks:
        return "DoS"
    elif label_str in probe_attacks:
        return "Probe"
    elif label_str in r2l_attacks:
        return "R2L"
    elif label_str in u2r_attacks:
        return "U2R"
    else:
        return "DoS"

test_labels_str = df_test_from_csv['label'].values

# 2.6 - Helper functions
def get_logit(model, x):
    """Returns the pre-sigmoid logit value."""
    out = x
    for i, layer in enumerate(model.net):
        if i == len(model.net) - 1:
            break
        out = layer(out)
    return out

def pgd_attack(model, x_original, modifiable_indices,
               feature_min, feature_max,
               epsilon, alpha=0.01, num_iter=40):
    x_adv = x_original.clone().detach()
    for _ in range(num_iter):
        x_adv = x_adv.clone().detach().requires_grad_(True)
        logit = get_logit(model, x_adv.unsqueeze(0))
        target = torch.tensor([[0.0]])
        loss = nn.BCEWithLogitsLoss()(logit, target)
        model.zero_grad()
        loss.backward()
        grad = x_adv.grad.data
        x_new = x_adv.detach().clone()
        for idx in modifiable_indices:
            x_new[idx] = x_adv[idx].detach() - alpha * grad[idx].sign()
        delta = x_new - x_original.detach()
        for idx in modifiable_indices:
            delta[idx] = torch.clamp(delta[idx], -epsilon, epsilon)
        x_new = x_original.detach() + delta
        x_new = torch.max(x_new, feature_min)
        x_new = torch.min(x_new, feature_max)
        x_adv = x_new.detach()
    return x_adv

# 2.7 - Full evaluation for all epsilon values
checker = AttackPlausibilityChecker(feature_names_ohe)
epsilons = [0.05, 0.1, 0.15, 0.2, 0.3]
alpha = 0.01

print("=" * 60)
print("TASK 2 - PGD RESULTS")
print("=" * 60)

results_task2 = {}

for epsilon in epsilons:
    successful_evasions = 0
    plausible_evasions = 0
    total = len(correctly_detected_attack_indices)
    
    for idx in correctly_detected_attack_indices:
        x_orig, y = test_dataset[idx]
        label_str = test_labels_str[idx]
        attack_type = get_attack_type(label_str)
        
        x_adv = pgd_attack(
            model=model,
            x_original=x_orig,
            modifiable_indices=modifiable_indices_extended,
            feature_min=feature_min,
            feature_max=feature_max,
            epsilon=epsilon,
            alpha=alpha,
            num_iter=40
        )
        
        with torch.no_grad():
            adv_pred = model(x_adv.unsqueeze(0)).item()
        
        if adv_pred < 0.5:
            successful_evasions += 1
            is_plausible, _ = checker.check_plausibility(
                x_adversarial=x_adv.numpy(),
                attack_type=attack_type,
                scaler=scaler
            )
            if is_plausible:
                plausible_evasions += 1
    
    results_task2[epsilon] = {
        "successful": successful_evasions,
        "plausible": plausible_evasions,
        "total": total
    }
    
    print(f"\nEpsilon = {epsilon}")
    print(f"  Total samples attacked:    {total}")
    print(f"  Successful evasions:       {successful_evasions} ({successful_evasions/total*100:.1f}%)")
    if successful_evasions > 0:
        rate = plausible_evasions / successful_evasions * 100
        print(f"  Plausible attacks:         {plausible_evasions} ({rate:.1f}%)")
    else:
        print(f"  Plausible attacks:         0")

print("\n" + "=" * 60)
print("Done!")

Number of features: 122
Modifiable features: 16
Correctly detected attacks: 8769
TASK 2 - PGD RESULTS

Epsilon = 0.05
  Total samples attacked:    8769
  Successful evasions:       129 (1.5%)
  Plausible attacks:         8 (6.2%)

Epsilon = 0.1
  Total samples attacked:    8769
  Successful evasions:       248 (2.8%)
  Plausible attacks:         40 (16.1%)

Epsilon = 0.15
  Total samples attacked:    8769
  Successful evasions:       340 (3.9%)
  Plausible attacks:         55 (16.2%)

Epsilon = 0.2
  Total samples attacked:    8769
  Successful evasions:       439 (5.0%)
  Plausible attacks:         84 (19.1%)

Epsilon = 0.3
  Total samples attacked:    8769
  Successful evasions:       541 (6.2%)
  Plausible attacks:         152 (28.1%)

Done!


In [33]:
# 2.8 - Model evaluation on test set
# Zero-day attacks: attacks not seen during training
train_labels_unique = set(df_train_from_csv['label'].unique())
zero_day_labels = set(df_test_from_csv['label'].unique()) - train_labels_unique
print(f"Zero-day attack types: {zero_day_labels}")

model.eval()
all_preds = []
all_targets = []
zero_day_preds = []
zero_day_targets = []
non_zero_day_preds = []
non_zero_day_targets = []

with torch.no_grad():
    for i, (features, label) in enumerate(test_dataset):
        pred = 1 if model(features).item() >= 0.5 else 0
        true_label = int(label.item())
        label_str = test_labels_str[i]

        all_preds.append(pred)
        all_targets.append(true_label)

        if label_str in zero_day_labels:
            zero_day_preds.append(pred)
            zero_day_targets.append(true_label)
        elif label_str != 'normal':
            non_zero_day_preds.append(pred)
            non_zero_day_targets.append(true_label)

print("\n" + "=" * 60)
print("TASK 2 - MODEL EVALUATION ON TEST SET")
print("=" * 60)

print("\nFull test set:")
print(f"  Accuracy:  {accuracy_score(all_targets, all_preds):.4f}")
print(f"  Precision: {precision_score(all_targets, all_preds):.4f}")
print(f"  Recall:    {recall_score(all_targets, all_preds):.4f}")
print(f"  F1-score:  {f1_score(all_targets, all_preds):.4f}")

print("\nZero-day attacks:")
print(f"  Accuracy:  {accuracy_score(zero_day_targets, zero_day_preds):.4f}")

print("\nNon zero-day attacks:")
print(f"  Accuracy:  {accuracy_score(non_zero_day_targets, non_zero_day_preds):.4f}")

Zero-day támadás típusok: {'sqlattack', 'httptunnel', 'mailbomb', 'udpstorm', 'snmpguess', 'xterm', 'worm', 'snmpgetattack', 'xlock', 'saint', 'processtable', 'mscan', 'sendmail', 'apache2', 'xsnoop', 'named', 'ps'}

2. FELADAT - MODELL ÉRTÉKELÉS A TESZT HALMAZON

Teljes teszt halmaz:
  Accuracy:  0.7900
  Precision: 0.9290
  Recall:    0.6833
  F1-score:  0.7874

Zero-day támadások:
  Accuracy:  0.4880

Nem zero-day támadások:
  Accuracy:  0.7640


# 3. feladat

In [53]:
# ============================================================
# 3. TASK - PGD WITH PLAUSIBILITY LOSS
# ============================================================

from sklearn.naive_bayes import GaussianNB
import numpy as np

# -----------------------------------------------------------
# 3.1 - Train GNB model (multi-class, attacks only)
# -----------------------------------------------------------

train_labels_original = df_train_from_csv['label'].values
attack_mask = train_labels_original != 'normal'
X_train_attacks = X_train[attack_mask]
y_train_attacks = train_labels_original[attack_mask]

gnb = GaussianNB()
gnb.fit(X_train_attacks, y_train_attacks)

print(f"GNB trained! Number of classes: {len(gnb.classes_)}")

# -----------------------------------------------------------
# 3.2 - Compute plausibility loss based on GNB
# -----------------------------------------------------------

def compute_plausibility_loss(x, attack_label, gnb):
    """
    Negative log-likelihood based on the GNB model.
    The smaller it is, the more x resembles the original attack type.
    L_plausibility = -log p(x | attack_label)
    """
    if attack_label not in gnb.classes_:
        return None
    
    class_idx = list(gnb.classes_).index(attack_label)
    
    # Extract μ and σ² from GNB
    mu = torch.tensor(gnb.theta_[class_idx], dtype=torch.float32)
    var = torch.tensor(gnb.var_[class_idx], dtype=torch.float32)
    var = torch.clamp(var, min=1e-6)
    
    # Gaussian log-likelihood: log p(xi|y) = -0.5*log(2π*σ²) - (xi-μi)²/(2σ²)
    log_likelihood = -0.5 * torch.log(2 * torch.tensor(np.pi) * var) \
                     - (x - mu) ** 2 / (2 * var)
    
    # Negative log-likelihood = loss
    return -log_likelihood.sum()

# -----------------------------------------------------------
# 3.3 - PGD with combined loss
# -----------------------------------------------------------

def pgd_attack_with_plausibility(model, x_original, attack_label,
                                  gnb, modifiable_indices,
                                  feature_min, feature_max,
                                  epsilon, alpha=0.01, num_iter=40,
                                  lambda_plausibility=0.1):
    """
    PGD attack with combined loss:
    L_total = L_classifier + λ * L_plausibility
    
    L_classifier: fools the IDS (pushes towards normal)
    L_plausibility: keeps it close to the original attack type (stays realistic)
    """
    x_adv = x_original.clone().detach()
    
    for _ in range(num_iter):
        x_adv = x_adv.clone().detach().requires_grad_(True)
        
        # L_classifier: fool the IDS
        logit = get_logit(model, x_adv.unsqueeze(0))
        target = torch.tensor([[0.0]])
        l_classifier = nn.BCEWithLogitsLoss()(logit, target)
        
        # L_plausibility: GNB-based loss
        l_plausibility = compute_plausibility_loss(x_adv, attack_label, gnb)
        
        # L_total = L_classifier + λ * L_plausibility
        l_total = l_classifier + lambda_plausibility * l_plausibility
        
        # Gradient
        model.zero_grad()
        l_total.backward()
        grad = x_adv.grad.data
        
        # Step only for modifiable features
        x_new = x_adv.detach().clone()
        for idx in modifiable_indices:
            x_new[idx] = x_adv[idx].detach() - alpha * grad[idx].sign()
        
        # Clamp 1: epsilon constraint
        delta = x_new - x_original.detach()
        for idx in modifiable_indices:
            delta[idx] = torch.clamp(delta[idx], -epsilon, epsilon)
        x_new = x_original.detach() + delta
        
        # Clamp 2: valid feature range
        x_new = torch.max(x_new, feature_min)
        x_new = torch.min(x_new, feature_max)
        
        x_adv = x_new.detach()
    
    return x_adv

# -----------------------------------------------------------
# 3.4 - Identify non-zero-day samples
# -----------------------------------------------------------

non_zero_day_attack_indices = []
for idx in correctly_detected_attack_indices:
    label_str = test_labels_str[idx]
    if label_str not in zero_day_labels:
        non_zero_day_attack_indices.append(idx)

print(f"Correctly detected NON-zero-day attacks: {len(non_zero_day_attack_indices)}")
print(f"Zero-day attacks (excluded): {len(correctly_detected_attack_indices) - len(non_zero_day_attack_indices)}")

# -----------------------------------------------------------
# 3.5 - Full evaluation for each epsilon value
# -----------------------------------------------------------

epsilons = [0.05, 0.1, 0.15, 0.2, 0.3]
alpha = 0.01
lambda_plausibility = 0.1

print("\n" + "=" * 60)
print("TASK 3 - PGD WITH PLAUSIBILITY LOSS")
print("=" * 60)

results_task3 = {}

for epsilon in epsilons:
    successful_evasions = 0
    plausible_evasions = 0
    total = len(non_zero_day_attack_indices)
    
    for idx in non_zero_day_attack_indices:
        x_orig, y = test_dataset[idx]
        label_str = test_labels_str[idx]
        attack_type = get_attack_type(label_str)
        
        x_adv = pgd_attack_with_plausibility(
            model=model,
            x_original=x_orig,
            attack_label=label_str,
            gnb=gnb,
            modifiable_indices=modifiable_indices_extended,
            feature_min=feature_min,
            feature_max=feature_max,
            epsilon=epsilon,
            alpha=alpha,
            num_iter=40,
            lambda_plausibility=lambda_plausibility
        )
        
        with torch.no_grad():
            adv_pred = model(x_adv.unsqueeze(0)).item()
        
        if adv_pred < 0.5:
            successful_evasions += 1
            is_plausible, _ = checker.check_plausibility(
                x_adversarial=x_adv.numpy(),
                attack_type=attack_type,
                scaler=scaler
            )
            if is_plausible:
                plausible_evasions += 1
    
    results_task3[epsilon] = {
        "successful": successful_evasions,
        "plausible": plausible_evasions,
        "total": total
    }
    
    print(f"\nEpsilon = {epsilon}")
    print(f"  Total attacked samples:    {total}")
    print(f"  Successful evasions:       {successful_evasions} ({successful_evasions/total*100:.1f}%)")
    if successful_evasions > 0:
        rate = plausible_evasions / successful_evasions * 100
        print(f"  Plausible attacks:         {plausible_evasions} ({rate:.1f}%)")
    else:
        print(f"  Plausible attacks:         0")

# -----------------------------------------------------------
# Comparison: Task 2 vs Task 3
# -----------------------------------------------------------

print("\n" + "=" * 75)
print("COMPARISON: Task 2 vs Task 3")
print("=" * 75)
print(f"{'Epsilon':<10} | {'2. Successful':>12} | {'2. Plausible':>15} | {'3. Successful':>12} | {'3. Plausible':>15}")
print("-" * 75)

for epsilon in epsilons:
    r2 = results_task2[epsilon]
    r3 = results_task3[epsilon]
    
    r2_siker = f"{r2['successful']} ({r2['successful']/r2['total']*100:.1f}%)"
    r3_siker = f"{r3['successful']} ({r3['successful']/r3['total']*100:.1f}%)"
    r2_plauz = f"{r2['plausible']/r2['successful']*100:.1f}%" if r2['successful'] > 0 else "0%"
    r3_plauz = f"{r3['plausible']/r3['successful']*100:.1f}%" if r3['successful'] > 0 else "0%"
    
    print(f"{epsilon:<10} | {r2_siker:>12} | {r2_plauz:>15} | {r3_siker:>12} | {r3_plauz:>15}")

print("=" * 75)
print("\nNote: Task 2 runs on 8769 samples, Task 3 runs on 6939 samples")
print("(zero-day attacks are excluded from Task 3)")

GNB trained! Number of classes: 22
Correctly detected NON-zero-day attacks: 6939
Zero-day attacks (excluded): 1830

TASK 3 - PGD WITH PLAUSIBILITY LOSS

Epsilon = 0.05
  Total attacked samples:    6939
  Successful evasions:       52 (0.7%)
  Plausible attacks:         6 (11.5%)

Epsilon = 0.1
  Total attacked samples:    6939
  Successful evasions:       132 (1.9%)
  Plausible attacks:         30 (22.7%)

Epsilon = 0.15
  Total attacked samples:    6939
  Successful evasions:       176 (2.5%)
  Plausible attacks:         35 (19.9%)

Epsilon = 0.2
  Total attacked samples:    6939
  Successful evasions:       227 (3.3%)
  Plausible attacks:         44 (19.4%)

Epsilon = 0.3
  Total attacked samples:    6939
  Successful evasions:       369 (5.3%)
  Plausible attacks:         90 (24.4%)

COMPARISON: Task 2 vs Task 3
Epsilon    | 2. Successful |    2. Plausible | 3. Successful |    3. Plausible
---------------------------------------------------------------------------
0.05       |   129